In [1]:
# Open the persistent DuckDB / DuckLake catalog created by the earlier notebooks.
# Run notebooks in README order from the assign1 folder.
import duckdb
from pathlib import Path
from IPython.display import display
Path("lakehouse_data").mkdir(exist_ok=True)
con = duckdb.connect("hotel_pipeline.duckdb")
con.execute("INSTALL ducklake;")
con.execute("LOAD ducklake;")
con.execute("ATTACH 'ducklake:hotel_catalog.duckdb' AS hotel_lake (DATA_PATH 'lakehouse_data/');")
print("Connected to the hotel lakehouse.")


Connected to the hotel lakehouse.


## Task 5: Gold Layer — Dimensional Modeling

### Star Schema Design

The Gold layer uses a star schema to support hotel booking analytics.

**Fact table:** fact_bookings

Grain: One row per validated hotel booking observation.

Measures:
- Booking count (1 per observation)
- Cancellation indicator
- Lead time in days
- Average daily rate (ADR)
- Total nights
- Total guests

**Dimension tables:**
- dim_hotel: Hotel category
- dim_date: Scheduled arrival date
- dim_market_segment: Booking market segment

### Design Justification

The fact table contains numeric measures and foreign keys connecting to dimensions. The dimensions provide descriptive attributes for grouping and filtering.

A source-record identifier is retained for traceability. Identical-looking booking observations remain separate because the original dataset does not contain unique booking IDs.

### Star-Schema Diagram (actual Gold table columns)

![Hotel Booking Star Schema](star_schema.png)

PK = primary key; FK = foreign key. Each dimension row may relate to many fact rows.


In [2]:
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.gold.dim_hotel AS
SELECT
    ROW_NUMBER() OVER (ORDER BY HotelType) AS hotel_key,
    HotelType AS hotel_type
FROM (
    SELECT DISTINCT
        CASE
            WHEN source_name = 'H1.csv' THEN 'Resort Hotel'
            WHEN source_name = 'H2.csv' THEN 'City Hotel'
        END AS HotelType
    FROM hotel_lake.silver.clean_bookings
)
""")

display(con.execute("""
SELECT * FROM hotel_lake.gold.dim_hotel
ORDER BY hotel_key
""").df())

,hotel_key,hotel_type
0,1,City Hotel
1,2,Resort Hotel


In [3]:
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.gold.dim_market_segment AS
SELECT
    ROW_NUMBER() OVER (ORDER BY market_segment)
        AS market_segment_key,
    market_segment
FROM (
    SELECT DISTINCT
        COALESCE(NULLIF(TRIM(MarketSegment), ''), 'Unknown')
            AS market_segment
    FROM hotel_lake.silver.clean_bookings
)
""")

display(con.execute("""
SELECT * FROM hotel_lake.gold.dim_market_segment
ORDER BY market_segment_key
""").df())

,market_segment_key,market_segment
0,1,Aviation
1,2,Complementary
2,3,Corporate
3,4,Direct
4,5,Groups
5,6,Offline TA/TO
6,7,Online TA
7,8,Undefined


In [4]:
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.gold.dim_date AS
WITH dates AS (
    SELECT DISTINCT
        MAKE_DATE(
            ArrivalDateYear,
            CASE ArrivalDateMonth
                WHEN 'January' THEN 1
                WHEN 'February' THEN 2
                WHEN 'March' THEN 3
                WHEN 'April' THEN 4
                WHEN 'May' THEN 5
                WHEN 'June' THEN 6
                WHEN 'July' THEN 7
                WHEN 'August' THEN 8
                WHEN 'September' THEN 9
                WHEN 'October' THEN 10
                WHEN 'November' THEN 11
                WHEN 'December' THEN 12
            END,
            ArrivalDateDayOfMonth
        ) AS arrival_date
    FROM hotel_lake.silver.clean_bookings
)
SELECT
    CAST(STRFTIME(arrival_date, '%Y%m%d') AS INTEGER)
        AS date_key,
    arrival_date,
    YEAR(arrival_date) AS year,
    MONTH(arrival_date) AS month_number,
    MONTHNAME(arrival_date) AS month_name,
    QUARTER(arrival_date) AS quarter
FROM dates
""")

display(con.execute("""
SELECT *
FROM hotel_lake.gold.dim_date
ORDER BY arrival_date
LIMIT 10
""").df())

,date_key,arrival_date,year,month_number,month_name,quarter
0,20150701,2015-07-01,2015,7,July,3
1,20150702,2015-07-02,2015,7,July,3
2,20150703,2015-07-03,2015,7,July,3
3,20150704,2015-07-04,2015,7,July,3
4,20150705,2015-07-05,2015,7,July,3
5,20150706,2015-07-06,2015,7,July,3
6,20150707,2015-07-07,2015,7,July,3
7,20150708,2015-07-08,2015,7,July,3
8,20150709,2015-07-09,2015,7,July,3
9,20150710,2015-07-10,2015,7,July,3


In [5]:
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.gold.fact_bookings AS
SELECT
    s.source_name,
    s.load_id,
    s.source_row_number,

    h.hotel_key,
    d.date_key,
    m.market_segment_key,

    1 AS booking_count,
    s.IsCanceled AS is_canceled,
    s.LeadTime AS lead_time,
    s.ADR AS adr,
    s.TotalNights AS total_nights,
    s.TotalGuests AS total_guests

FROM hotel_lake.silver.clean_bookings s

JOIN hotel_lake.gold.dim_hotel h
    ON h.hotel_type =
        CASE
            WHEN s.source_name = 'H1.csv' THEN 'Resort Hotel'
            WHEN s.source_name = 'H2.csv' THEN 'City Hotel'
        END

JOIN hotel_lake.gold.dim_date d
    ON d.date_key = CAST(
        STRFTIME(
            MAKE_DATE(
                s.ArrivalDateYear,
                CASE s.ArrivalDateMonth
                    WHEN 'January' THEN 1
                    WHEN 'February' THEN 2
                    WHEN 'March' THEN 3
                    WHEN 'April' THEN 4
                    WHEN 'May' THEN 5
                    WHEN 'June' THEN 6
                    WHEN 'July' THEN 7
                    WHEN 'August' THEN 8
                    WHEN 'September' THEN 9
                    WHEN 'October' THEN 10
                    WHEN 'November' THEN 11
                    WHEN 'December' THEN 12
                END,
                s.ArrivalDateDayOfMonth
            ),
            '%Y%m%d'
        ) AS INTEGER
    )

JOIN hotel_lake.gold.dim_market_segment m
    ON m.market_segment =
        COALESCE(NULLIF(TRIM(s.MarketSegment), ''), 'Unknown')
""")

print("Gold fact table created successfully!")

Gold fact table created successfully!


In [6]:
gold_report = con.execute("""
SELECT
    (SELECT COUNT(*)
     FROM hotel_lake.silver.clean_bookings)
        AS silver_rows,

    (SELECT COUNT(*)
     FROM hotel_lake.gold.fact_bookings)
        AS gold_fact_rows,

    (SELECT COUNT(*)
     FROM hotel_lake.gold.dim_hotel)
        AS hotel_dimension_rows,

    (SELECT COUNT(*)
     FROM hotel_lake.gold.dim_date)
        AS date_dimension_rows,

    (SELECT COUNT(*)
     FROM hotel_lake.gold.dim_market_segment)
        AS market_segment_dimension_rows
""").df()

display(gold_report)

assert (
    gold_report["silver_rows"].iloc[0]
    == gold_report["gold_fact_rows"].iloc[0]
)

print("Gold row-count reconciliation passed!")

,silver_rows,gold_fact_rows,hotel_dimension_rows,date_dimension_rows,market_segment_dimension_rows
0,119209,119209,2,793,8


Gold row-count reconciliation passed!


### Gold Layer Integrity Validation

This section verifies that the Gold fact table has unique source-record identifiers and that all foreign keys match their corresponding dimension tables.


In [7]:
integrity_report = con.execute("""
SELECT
    COUNT(*) AS total_fact_rows,

    COUNT(DISTINCT (
        source_name, load_id, source_row_number
    )) AS unique_booking_records,

    SUM(CASE WHEN h.hotel_key IS NULL
        THEN 1 ELSE 0 END) AS invalid_hotel_keys,

    SUM(CASE WHEN d.date_key IS NULL
        THEN 1 ELSE 0 END) AS invalid_date_keys,

    SUM(CASE WHEN m.market_segment_key IS NULL
        THEN 1 ELSE 0 END) AS invalid_segment_keys

FROM hotel_lake.gold.fact_bookings f

LEFT JOIN hotel_lake.gold.dim_hotel h
    ON f.hotel_key = h.hotel_key

LEFT JOIN hotel_lake.gold.dim_date d
    ON f.date_key = d.date_key

LEFT JOIN hotel_lake.gold.dim_market_segment m
    ON f.market_segment_key = m.market_segment_key
""").df()

display(integrity_report)

assert integrity_report["total_fact_rows"].iloc[0] == 119209
assert integrity_report["unique_booking_records"].iloc[0] == 119209

for col in [
    "invalid_hotel_keys",
    "invalid_date_keys",
    "invalid_segment_keys"
]:
    assert integrity_report[col].iloc[0] == 0

print("Gold integrity validation passed!")

,total_fact_rows,unique_booking_records,invalid_hotel_keys,invalid_date_keys,invalid_segment_keys
0,119209,119209,0.0,0.0,0.0


Gold integrity validation passed!
